# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [4]:
# TODO: Fill this in before submitting.
GROUP_ID = 9
NOTEBOOK_URL = "https://github.com/DonAlberton/tbd-workshop-1/blob/master/notebooks/tbd_phase_2.ipynb"
GROUP_MEMBERS = [
    "Alberto Szpejewski / 318843",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "DonAlberton" in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [1]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.13.3
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 16
RAM GiB: 31.11


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [5]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Banking-like transactions',
 'feature': 'risk flags',
 'stress': 'selective filters, top-k, and sorting'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [ ]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "small"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 9 {'name': 'Banking-like transactions', 'feature': 'risk flags', 'stress': 'selective filters, top-k, and sorting'}
Rows: 2000000
Run seed recorded in manifest: 65561328993725911168030411239838663008
Output directory: ..\data\phase2_26L\group_09


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [ ]:
BANKING_RISK_FLAGS = [
    "velocity_alert", "geo_mismatch", "unusual_amount", "new_device",
    "after_hours", "foreign_ip", "multiple_attempts", "high_value",
    "dormant_account", "blocked_country",
]

TX_TYPE_MAP = {
    "A": "purchase", "B": "transfer", "C": "withdrawal",
    "D": "deposit",  "E": "refund",   "F": "fee",
}

# 95th pct = 0.40 → 5% of rows are suspicious+confirmed
# 99th pct = 0.55 → 1% of rows are confirmed_fraud
_FLAGGED_THRESHOLD = 0.40
_CONFIRMED_THRESHOLD = 0.55


def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, rng):
    n = df.height

    # With 2M rows: 5% exceed _FLAGGED_THRESHOLD, 1% exceed _CONFIRMED_THRESHOLD.
    risk_score = rng.beta(a=1.5, b=8.0, size=n).round(4)

    fraud_label = np.where(
        risk_score > _CONFIRMED_THRESHOLD, "confirmed_fraud",
        np.where(risk_score > _FLAGGED_THRESHOLD, "suspicious", "clean"),
    )
    is_flagged = risk_score > _FLAGGED_THRESHOLD

    channel = rng.choice(
        ["mobile_app", "web", "atm", "branch", "api"],
        size=n, p=[0.45, 0.25, 0.15, 0.10, 0.05],
    )
    currency = rng.choice(
        ["EUR", "USD", "GBP", "PLN", "CHF"],
        size=n, p=[0.35, 0.30, 0.10, 0.15, 0.10],
    )
    account_type = rng.choice(
        ["personal", "business", "savings"],
        size=n, p=[0.60, 0.25, 0.15],
    )
    merchant_category = rng.choice(
        ["retail", "food", "travel", "entertainment", "utilities", "healthcare"],
        size=n, p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10],
    )
    
    merchant_id = skewed_ids(rng, n, max_id=1_000, hot_fraction=0.05, hot_probability=0.60)
    risk_flags = random_tag_lists(rng, n, vocabulary=BANKING_RISK_FLAGS, min_tags=0, max_tags=3)

    return (
        df
        .drop(["device", "tags"])
        .rename({
            "event_id":  "transaction_id",
            "entity_id": "account_id",
            "category":  "transaction_type",
            "metric_1":  "amount",
            "metric_2":  "balance_snapshot",
        })
        .with_columns([
            pl.col("transaction_type").replace(TX_TYPE_MAP),
            pl.Series("risk_score",        risk_score),
            pl.Series("is_flagged",        is_flagged),
            pl.Series("fraud_label",       fraud_label).cast(pl.Categorical),
            pl.Series("channel",           channel).cast(pl.Categorical),
            pl.Series("currency",          currency).cast(pl.Categorical),
            pl.Series("account_type",      account_type).cast(pl.Categorical),
            pl.Series("merchant_category", merchant_category).cast(pl.Categorical),
            pl.Series("merchant_id",       merchant_id),
            pl.Series("risk_flags",        risk_flags),
        ])
    )


def generate_dimension_table(rng):
    n_merchants = 1_000
    return pl.DataFrame({
        "merchant_id":       np.arange(1, n_merchants + 1),
        "merchant_name":     [fake.company() for _ in range(n_merchants)],
        "merchant_category": rng.choice(
            ["retail", "food", "travel", "entertainment", "utilities", "healthcare"],
            size=n_merchants, p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10],
        ),
        "merchant_country":  rng.choice(
            ["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n_merchants,
        ),
        "risk_tier":         rng.choice(
            ["low", "medium", "high"], size=n_merchants, p=[0.70, 0.20, 0.10],
        ),
        "avg_txn_value":     rng.lognormal(mean=3.5, sigma=0.8, size=n_merchants).round(2),
        "is_blacklisted":    rng.random(n_merchants) < 0.02,
    })

In [ ]:
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

# Default layout: random order, one file - baseline for all benchmarks
events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Partitioned layout: one file per calendar day - file-level pruning
# when queries filter on event_date.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# Optimized layout for Q1 (fraud_label = 'confirmed_fraud' filter)
events.sort("risk_score", descending=True).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)


(
    events
    .select([
        "transaction_id", "account_id", "country", "merchant_category",
        "amount", "risk_score", "is_flagged", "fraud_label",
    ])
    .write_csv(CSV_EVENTS_PATH)
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events":             str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized":   str(OPTIMIZED_EVENTS_PATH),
        "dimension":          str(DIMENSION_PATH),
        "events_csv_q1":      str(CSV_EVENTS_PATH),
    },
    "schema": {col: str(dtype) for col, dtype in zip(events.columns, events.dtypes)},
    "environment": {
        "python":            platform.python_version(),
        "polars":            pl.__version__,
        "pandas":            pd.__version__,
        "duckdb":            duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib":           round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))

{
  "created_at_utc": "2026-05-31T19:42:56.289230+00:00",
  "group_id": 9,
  "variant": {
    "name": "Banking-like transactions",
    "feature": "risk flags",
    "stress": "selective filters, top-k, and sorting"
  },
  "scale": "small",
  "rows": 2000000,
  "run_seed": 65561328993725911168030411239838663008,
  "paths": {
    "events": "..\\data\\phase2_26L\\group_09\\events.parquet",
    "events_partitioned": "..\\data\\phase2_26L\\group_09\\events_partitioned",
    "events_optimized": "..\\data\\phase2_26L\\group_09\\events_optimized.parquet",
    "dimension": "..\\data\\phase2_26L\\group_09\\dimension.parquet",
    "events_csv_q1": "..\\data\\phase2_26L\\group_09\\events.csv"
  },
  "schema": {
    "transaction_id": "Int64",
    "account_id": "Int64",
    "event_ts": "Datetime(time_unit='us', time_zone=None)",
    "transaction_type": "String",
    "country": "String",
    "amount": "Float64",
    "balance_snapshot": "Int64",
    "event_date": "Date",
    "risk_score": "Float64",
  

## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [13]:
print("=== Schema ===")
print(events.schema)
print(f"\nRows: {events.height:,}   Columns: {events.width}")

print("\n=== Null counts ===")
print(events.null_count())

print("\n=== transaction_type distribution ===")
print(events["transaction_type"].value_counts(sort=True))

print("\n=== fraud_label distribution ===")
print(events["fraud_label"].value_counts(sort=True))

print("\n=== channel distribution ===")
print(events["channel"].value_counts(sort=True))

print("\n=== currency distribution ===")
print(events["currency"].value_counts(sort=True))

print("\n=== risk_score statistics ===")
print(events["risk_score"].describe())

print("\n=== amount statistics ===")
print(events["amount"].describe())

print(
    f"\n=== is_flagged: {events['is_flagged'].sum():,} flagged "
    f"({events['is_flagged'].mean() * 100:.1f}% of rows) ==="
)

print("\n=== Top-5 accounts by tx count (skew check) ===")
print(
    events.group_by("account_id")
    .agg(pl.len().alias("tx_count"))
    .sort("tx_count", descending=True)
    .head(5)
)

print("\n=== Dimension table schema ===")
print(dimension.schema)
print(dimension["risk_tier"].value_counts(sort=True))
print(f"Blacklisted merchants: {dimension['is_blacklisted'].sum()}")

=== Schema ===
Schema({'transaction_id': Int64, 'account_id': Int64, 'event_ts': Datetime(time_unit='us', time_zone=None), 'transaction_type': String, 'country': String, 'amount': Float64, 'balance_snapshot': Int64, 'event_date': Date, 'risk_score': Float64, 'is_flagged': Boolean, 'fraud_label': Categorical, 'channel': Categorical, 'currency': Categorical, 'account_type': Categorical, 'merchant_category': Categorical, 'merchant_id': Int64, 'risk_flags': List(String)})

Rows: 2,000,000   Columns: 17

=== Null counts ===
shape: (1, 17)
┌───────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ transacti ┆ account_i ┆ event_ts ┆ transacti ┆ … ┆ account_t ┆ merchant_ ┆ merchant_ ┆ risk_flag │
│ on_id     ┆ d         ┆ ---      ┆ on_type   ┆   ┆ ype       ┆ category  ┆ id        ┆ s         │
│ ---       ┆ ---       ┆ u32      ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32       ┆ u32       ┆          ┆ u32       ┆   ┆ u32

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [ ]:
import statistics

N_REPS = 5

BENCHMARK_COLUMNS = [
    "library_engine", "mode", "query_name", "data_format",
    "layout", "rows", "median_time_s", "peak_memory_mb",
    "input_size_mb", "result_check", "notes",
]

benchmark_results = []


def file_size_mb(path):
    p = Path(path)
    if p.is_dir():
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
    return p.stat().st_size / 1e6 if p.exists() else 0.0


def benchmark(fn, reps=N_REPS):
    """
    Run fn() `reps` times for timing, then one dedicated run for peak-memory delta.
    Returns (median_s, peak_delta_mb, last_result).

    Limitation: all runs share one kernel process. Previous allocations and
    engine-internal caches are NOT cleared. gc.collect() is called before each
    timing rep (Python GC only). For accurate memory comparisons, see Task 3.1.
    """
    times, last = [], None
    for _ in range(reps):
        gc.collect()
        t0 = time.perf_counter()
        last = fn()
        times.append(time.perf_counter() - t0)
    gc.collect()
    mem_samples, last = memory_usage(fn, interval=0.05, retval=True)
    peak_delta = max(mem_samples) - min(mem_samples)
    return statistics.median(times), peak_delta, last


def record(library_engine, mode, query_name, data_format, layout,
           rows, median_time_s, peak_memory_mb, input_size_mb,
           result_check, notes=""):
    benchmark_results.append({
        "library_engine": library_engine,
        "mode":           mode,
        "query_name":     query_name,
        "data_format":    data_format,
        "layout":         layout,
        "rows":           rows,
        "median_time_s":  round(median_time_s, 4),
        "peak_memory_mb": round(peak_memory_mb, 1),
        "input_size_mb":  round(input_size_mb, 1),
        "result_check":   result_check,
        "notes":          notes,
    })


EVENTS_SIZE_MB = file_size_mb(EVENTS_PATH)
OPT_SIZE_MB    = file_size_mb(OPTIMIZED_EVENTS_PATH)
CSV_SIZE_MB    = file_size_mb(CSV_EVENTS_PATH)
DIM_SIZE_MB    = file_size_mb(DIMENSION_PATH)

print(f"Benchmark helper ready - {N_REPS} repetitions per run")
print(f"  events.parquet            {EVENTS_SIZE_MB:.1f} MB")
print(f"  events_optimized.parquet  {OPT_SIZE_MB:.1f} MB")
print(f"  events.csv (Q1 cols)      {CSV_SIZE_MB:.1f} MB")
print(f"  dimension.parquet         {DIM_SIZE_MB:.1f} MB")

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [ ]:
QUERY_SPECS = {
    "Q1_fraud_by_country": {
        "class": "selective filter + aggregation",
        "description": (
            "Filter to confirmed_fraud rows (-1% selectivity), then aggregate "
            "total amount and average risk score by country and merchant_category."
        ),
        "hypothesis_best": (
            "DuckDB or Polars lazy - both push the fraud_label predicate into "
            "the Parquet reader and skip row groups where max(risk_score) < 0.90."
        ),
        "hypothesis_memory": (
            "Pandas default - reads all rows and all columns into memory before filtering."
        ),
        "layout_help": (
            "Optimized Parquet sorted by risk_score DESC with row_group_size=100_000: "
            "flagged rows cluster in the first few row groups, the rest can be skipped."
        ),
        "sql": """
            SELECT country, merchant_category,
                   COUNT(*)        AS n_confirmed,
                   SUM(amount)     AS total_amount,
                   AVG(risk_score) AS avg_risk
            FROM   events
            WHERE  fraud_label = 'confirmed_fraud'
            GROUP BY country, merchant_category
            ORDER BY total_amount DESC
        """,
    },
    "Q2_top_suspicious_accounts": {
        "class": "high-cardinality group-by + top-k + sorting",
        "description": (
            "Filter to is_flagged rows (-5%), group by account_id (up to 200K distinct values), "
            "sum suspicious amounts, return the top-20 accounts."
        ),
        "hypothesis_best": (
            "DuckDB or Polars - in-process hash aggregation, no shuffle. "
            "The 200K-key group-by should fit in RAM easily."
        ),
        "hypothesis_memory": (
            "PySpark local - shuffle on 200K keys writes spill data to disk."
        ),
        "layout_help": (
            "No layout helps this group-by. The skewed account_id distribution "
            "stresses the engine hash-table (top 2% of accounts hold 50% of rows)."
        ),
        "sql": """
            SELECT account_id,
                   COUNT(*)        AS suspicious_tx_count,
                   SUM(amount)     AS total_suspicious_amount,
                   MAX(risk_score) AS max_risk
            FROM   events
            WHERE  is_flagged = TRUE
            GROUP BY account_id
            ORDER BY total_suspicious_amount DESC
            LIMIT 20
        """,
    },
    "Q3_high_risk_merchant_join": {
        "class": "join with dimension table + aggregation",
        "description": (
            "Join the events table with the 1000-row merchant dimension on merchant_id, "
            "filter to high-risk or blacklisted merchants (-12% of merchants), "
            "aggregate by account_type, currency, and risk_tier."
        ),
        "hypothesis_best": (
            "DuckDB - broadcast hash join with a 1000-row probe side is trivial; "
            "Polars lazy is a close second."
        ),
        "hypothesis_memory": (
            "Pandas - materializes the full join result (all matched rows) before "
            "the aggregation reduces it."
        ),
        "layout_help": (
            "Dimension is small enough to broadcast in every engine; "
            "no special layout needed."
        ),
        "sql": """
            SELECT e.account_type, e.currency, m.risk_tier,
                   COUNT(*)          AS n_tx,
                   SUM(e.amount)     AS total_amount,
                   AVG(e.risk_score) AS avg_risk
            FROM   events   e
            JOIN   dimension m ON e.merchant_id = m.merchant_id
            WHERE  m.risk_tier = 'high'
               OR  m.is_blacklisted = TRUE
            GROUP BY e.account_type, e.currency, m.risk_tier
            ORDER BY total_amount DESC
        """,
    },
}

for name, spec in QUERY_SPECS.items():
    print(f"\n{'='*60}")
    print(f"Query:            {name}")
    print(f"  Class:          {spec['class']}")
    print(f"  Description:    {spec['description']}")
    print(f"  Expected best:  {spec['hypothesis_best']}")
    print(f"  Expected memory:{spec['hypothesis_memory']}")
    print(f"  Layout helps:   {spec['layout_help']}")


Query:            Q1_fraud_by_country
  Class:          selective filter + aggregation
  Description:    Filter to confirmed_fraud rows (~1% selectivity), then aggregate total amount and average risk score by country and merchant_category.
  Expected best:  DuckDB or Polars lazy — both push the fraud_label predicate into the Parquet reader and skip row groups where max(risk_score) < 0.90.
  Expected memory:Pandas default — reads all rows and all columns into memory before filtering.
  Layout helps:   Optimized Parquet sorted by risk_score DESC with row_group_size=100_000: flagged rows cluster in the first few row groups, the rest can be skipped.

Query:            Q2_top_suspicious_accounts
  Class:          high-cardinality group-by + top-k + sorting
  Description:    Filter to is_flagged rows (~5%), group by account_id (up to 200K distinct values), sum suspicious amounts, return the top-20 accounts.
  Expected best:  DuckDB or Polars — in-process hash aggregation, no shuffle. The 20

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [ ]:
spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", str(psutil.cpu_count(logical=True)))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version       :", spark.version)
print("Master              :", spark.sparkContext.master)
print("Default parallelism :", spark.sparkContext.defaultParallelism)

In [ ]:
# Pandas - both dtype backends, three queries
# NOTE: Pandas reads ALL columns by default (no column pruning).
# This is intentional - it shows the overhead vs. DuckDB/Polars lazy.

print("=== Pandas dtype comparison ===")
_np = pd.read_parquet(EVENTS_PATH)
_pa = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
for col in ["transaction_type", "fraud_label", "is_flagged", "amount", "risk_score"]:
    print(f"  {col:20}  numpy={_np[col].dtype}  pyarrow={_pa[col].dtype}")
del _np, _pa

_BACKENDS = [
    ("numpy",   {}),
    ("pyarrow", {"engine": "pyarrow", "dtype_backend": "pyarrow"}),
]

for backend, rk in _BACKENDS:
    label = f"pandas-{backend}"

    def _q1(rk=rk):
        df = pd.read_parquet(EVENTS_PATH, **rk)
        return (
            df[df["fraud_label"] == "confirmed_fraud"]
            .groupby(["country", "merchant_category"])
            .agg(n_confirmed=("transaction_id", "count"),
                 total_amount=("amount", "sum"),
                 avg_risk=("risk_score", "mean"))
            .sort_values("total_amount", ascending=False)
            .reset_index()
        )

    def _q2(rk=rk):
        df = pd.read_parquet(EVENTS_PATH, **rk)
        return (
            df[df["is_flagged"]]
            .groupby("account_id")
            .agg(suspicious_tx_count=("transaction_id", "count"),
                 total_suspicious_amount=("amount", "sum"),
                 max_risk=("risk_score", "max"))
            .sort_values("total_suspicious_amount", ascending=False)
            .head(20)
            .reset_index()
        )

    def _q3(rk=rk):
        df  = pd.read_parquet(EVENTS_PATH, **rk)
        dim = pd.read_parquet(DIMENSION_PATH, **rk)
        hr  = dim[(dim["risk_tier"] == "high") | dim["is_blacklisted"]]
        m   = df.merge(hr[["merchant_id", "risk_tier"]], on="merchant_id")
        return (
            m.groupby(["account_type", "currency", "risk_tier"])
            .agg(n_tx=("transaction_id", "count"),
                 total_amount=("amount", "sum"),
                 avg_risk=("risk_score", "mean"))
            .sort_values("total_amount", ascending=False)
            .reset_index()
        )

    for qname, fn in [("Q1_fraud_by_country",        _q1),
                      ("Q2_top_suspicious_accounts", _q2),
                      ("Q3_high_risk_merchant_join", _q3)]:
        t, mem, res = benchmark(fn)
        record(label, "eager", qname, "parquet", "default",
               N_ROWS, t, mem, EVENTS_SIZE_MB, f"rows={len(res)}", backend)
        print(f"{label:20}  {qname:35}  {t:.3f}s  +{mem:.0f}MB  rows={len(res)}")

pd.DataFrame(benchmark_results).sort_values(["query_name", "library_engine"])

In [ ]:
# Polars - eager, lazy, lazy+streaming - three queries

def pol_q1_eager():
    return (pl.read_parquet(EVENTS_PATH)
            .filter(pl.col("fraud_label") == "confirmed_fraud")
            .group_by(["country", "merchant_category"])
            .agg(pl.len().alias("n_confirmed"),
                 pl.col("amount").sum().alias("total_amount"),
                 pl.col("risk_score").mean().alias("avg_risk"))
            .sort("total_amount", descending=True))

def pol_q1_lazy():
    return (pl.scan_parquet(EVENTS_PATH)
            .filter(pl.col("fraud_label") == "confirmed_fraud")
            .group_by(["country", "merchant_category"])
            .agg(pl.len().alias("n_confirmed"),
                 pl.col("amount").sum().alias("total_amount"),
                 pl.col("risk_score").mean().alias("avg_risk"))
            .sort("total_amount", descending=True)
            .collect())

def pol_q1_stream():
    return (pl.scan_parquet(EVENTS_PATH)
            .filter(pl.col("fraud_label") == "confirmed_fraud")
            .group_by(["country", "merchant_category"])
            .agg(pl.len().alias("n_confirmed"),
                 pl.col("amount").sum().alias("total_amount"),
                 pl.col("risk_score").mean().alias("avg_risk"))
            .sort("total_amount", descending=True)
            .collect(engine="streaming"))

def pol_q2_eager():
    return (pl.read_parquet(EVENTS_PATH)
            .filter(pl.col("is_flagged"))
            .group_by("account_id")
            .agg(pl.len().alias("suspicious_tx_count"),
                 pl.col("amount").sum().alias("total_suspicious_amount"),
                 pl.col("risk_score").max().alias("max_risk"))
            .sort("total_suspicious_amount", descending=True)
            .head(20))

def pol_q2_lazy():
    return (pl.scan_parquet(EVENTS_PATH)
            .filter(pl.col("is_flagged"))
            .group_by("account_id")
            .agg(pl.len().alias("suspicious_tx_count"),
                 pl.col("amount").sum().alias("total_suspicious_amount"),
                 pl.col("risk_score").max().alias("max_risk"))
            .sort("total_suspicious_amount", descending=True)
            .head(20)
            .collect())

def pol_q2_stream():
    return (pl.scan_parquet(EVENTS_PATH)
            .filter(pl.col("is_flagged"))
            .group_by("account_id")
            .agg(pl.len().alias("suspicious_tx_count"),
                 pl.col("amount").sum().alias("total_suspicious_amount"),
                 pl.col("risk_score").max().alias("max_risk"))
            .sort("total_suspicious_amount", descending=True)
            .head(20)
            .collect(engine="streaming"))

def pol_q3_eager():
    dim = pl.read_parquet(DIMENSION_PATH)
    hr  = dim.filter((pl.col("risk_tier") == "high") | pl.col("is_blacklisted"))
    return (pl.read_parquet(EVENTS_PATH)
            .join(hr.select(["merchant_id", "risk_tier"]), on="merchant_id")
            .group_by(["account_type", "currency", "risk_tier"])
            .agg(pl.len().alias("n_tx"),
                 pl.col("amount").sum().alias("total_amount"),
                 pl.col("risk_score").mean().alias("avg_risk"))
            .sort("total_amount", descending=True))

def pol_q3_lazy():
    hr = (pl.scan_parquet(DIMENSION_PATH)
          .filter((pl.col("risk_tier") == "high") | pl.col("is_blacklisted"))
          .select(["merchant_id", "risk_tier"]))
    return (pl.scan_parquet(EVENTS_PATH)
            .join(hr, on="merchant_id")
            .group_by(["account_type", "currency", "risk_tier"])
            .agg(pl.len().alias("n_tx"),
                 pl.col("amount").sum().alias("total_amount"),
                 pl.col("risk_score").mean().alias("avg_risk"))
            .sort("total_amount", descending=True)
            .collect())

def pol_q3_stream():
    hr = (pl.scan_parquet(DIMENSION_PATH)
          .filter((pl.col("risk_tier") == "high") | pl.col("is_blacklisted"))
          .select(["merchant_id", "risk_tier"]))
    return (pl.scan_parquet(EVENTS_PATH)
            .join(hr, on="merchant_id")
            .group_by(["account_type", "currency", "risk_tier"])
            .agg(pl.len().alias("n_tx"),
                 pl.col("amount").sum().alias("total_amount"),
                 pl.col("risk_score").mean().alias("avg_risk"))
            .sort("total_amount", descending=True)
            .collect(engine="streaming"))

_pol_queries = [
    ("Q1_fraud_by_country",        pol_q1_eager, pol_q1_lazy, pol_q1_stream),
    ("Q2_top_suspicious_accounts", pol_q2_eager, pol_q2_lazy, pol_q2_stream),
    ("Q3_high_risk_merchant_join", pol_q3_eager, pol_q3_lazy, pol_q3_stream),
]
for qname, *fns in _pol_queries:
    for mode, idx in [("eager", 0), ("lazy", 1), ("streaming", 2)]:
        fn = fns[idx]
        t, mem, res = benchmark(fn)
        record("polars", mode, qname, "parquet", "default",
               N_ROWS, t, mem, EVENTS_SIZE_MB, f"rows={len(res)}")
        print(f"polars-{mode:10}  {qname:35}  {t:.3f}s  +{mem:.0f}MB  rows={len(res)}")

pd.DataFrame(benchmark_results).sort_values(["query_name", "library_engine"]).tail(9)

In [ ]:
# DuckDB - SQL directly on Parquet, full column pruning + predicate pushdown

_con = duckdb.connect()
_con.execute(f"CREATE VIEW events    AS SELECT * FROM read_parquet('{EVENTS_PATH}')")
_con.execute(f"CREATE VIEW dimension AS SELECT * FROM read_parquet('{DIMENSION_PATH}')")

_DDB = {
    "Q1_fraud_by_country": """
        SELECT country, merchant_category,
               COUNT(*) AS n_confirmed, SUM(amount) AS total_amount, AVG(risk_score) AS avg_risk
        FROM events WHERE fraud_label = 'confirmed_fraud'
        GROUP BY country, merchant_category ORDER BY total_amount DESC
    """,
    "Q2_top_suspicious_accounts": """
        SELECT account_id, COUNT(*) AS suspicious_tx_count,
               SUM(amount) AS total_suspicious_amount, MAX(risk_score) AS max_risk
        FROM events WHERE is_flagged = TRUE
        GROUP BY account_id ORDER BY total_suspicious_amount DESC LIMIT 20
    """,
    "Q3_high_risk_merchant_join": """
        SELECT e.account_type, e.currency, m.risk_tier,
               COUNT(*) AS n_tx, SUM(e.amount) AS total_amount, AVG(e.risk_score) AS avg_risk
        FROM events e JOIN dimension m ON e.merchant_id = m.merchant_id
        WHERE m.risk_tier = 'high' OR m.is_blacklisted = TRUE
        GROUP BY e.account_type, e.currency, m.risk_tier ORDER BY total_amount DESC
    """,
}

for qname, sql in _DDB.items():
    def fn(s=sql): return _con.execute(s).fetchdf()
    t, mem, res = benchmark(fn)
    record("duckdb", "sql", qname, "parquet", "default",
           N_ROWS, t, mem, EVENTS_SIZE_MB, f"rows={len(res)}")
    print(f"duckdb  {qname:35}  {t:.3f}s  +{mem:.0f}MB  rows={len(res)}")

_con.close()
pd.DataFrame(benchmark_results).sort_values(["query_name", "library_engine"]).tail(3)

In [ ]:
from pyspark.sql import functions as F

events_sdf = spark.read.parquet(str(EVENTS_PATH))
dim_sdf    = spark.read.parquet(str(DIMENSION_PATH))

# Warm up: trigger JVM class loading and Parquet schema read
events_sdf.filter(F.col("is_flagged") == True).count()

# Cache small dimension table for join (broadcast happens automatically, but cache avoids re-read)
hr_dim = (dim_sdf
          .filter((F.col("risk_tier") == "high") | (F.col("is_blacklisted") == True))
          .select("merchant_id", "risk_tier")
          .cache())
hr_dim.count()  # materialize cache

def spark_q1():
    return (events_sdf
            .filter(F.col("fraud_label") == "confirmed_fraud")
            .groupBy("country", "merchant_category")
            .agg(F.count("*").alias("n_confirmed"),
                 F.sum("amount").alias("total_amount"),
                 F.avg("risk_score").alias("avg_risk"))
            .orderBy(F.desc("total_amount"))
            .collect())

def spark_q2():
    return (events_sdf
            .filter(F.col("is_flagged") == True)
            .groupBy("account_id")
            .agg(F.count("*").alias("suspicious_tx_count"),
                 F.sum("amount").alias("total_suspicious_amount"),
                 F.max("risk_score").alias("max_risk"))
            .orderBy(F.desc("total_suspicious_amount"))
            .limit(20)
            .collect())

def spark_q3():
    return (events_sdf
            .join(hr_dim, on="merchant_id", how="inner")
            .groupBy("account_type", "currency", "risk_tier")
            .agg(F.count("*").alias("n_tx"),
                 F.sum("amount").alias("total_amount"),
                 F.avg("risk_score").alias("avg_risk"))
            .orderBy(F.desc("total_amount"))
            .collect())

for qname, fn in [("Q1_fraud_by_country",        spark_q1),
                  ("Q2_top_suspicious_accounts", spark_q2),
                  ("Q3_high_risk_merchant_join", spark_q3)]:
    t, mem, res = benchmark(fn, reps=3)  # fewer reps: Spark has high per-run overhead
    record("pyspark-local", "sql", qname, "parquet", "default",
           N_ROWS, t, mem, EVENTS_SIZE_MB, f"rows={len(res)}",
           f"local[*] cores={spark.sparkContext.defaultParallelism}")
    print(f"pyspark-local  {qname:35}  {t:.3f}s  +{mem:.0f}MB  rows={len(res)}")

hr_dim.unpersist()

print("\n=== Full benchmark results ===")
pd.DataFrame(benchmark_results).sort_values(["query_name", "median_time_s"])

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [ ]:
# Task 2.5: Q1 on three physical layouts with DuckDB
# Q1: WHERE fraud_label = 'confirmed_fraud' GROUP BY country, merchant_category
#
# Why sorting by risk_score DESC helps Q1:
#   The optimized file groups all high-risk rows (risk_score > _CONFIRMED_THRESHOLD)
#   into the first few row groups. With row_group_size=100_000, the reader's
#   min/max statistics on risk_score allow skipping every row group where
#   max(risk_score) < _CONFIRMED_THRESHOLD - typically 95-99% of row groups.
#   CSV has no such structure: every row and every column must be read.

_Q1_SQL = """
SELECT country, merchant_category,
       COUNT(*) AS n_confirmed, SUM(amount) AS total_amount, AVG(risk_score) AS avg_risk
FROM read_parquet('{path}')
WHERE fraud_label = 'confirmed_fraud'
GROUP BY country, merchant_category ORDER BY total_amount DESC
"""

_Q1_CSV = """
SELECT country, merchant_category,
       COUNT(*) AS n_confirmed, SUM(amount) AS total_amount, AVG(risk_score) AS avg_risk
FROM read_csv_auto('{path}')
WHERE fraud_label = 'confirmed_fraud'
GROUP BY country, merchant_category ORDER BY total_amount DESC
"""

layout_rows = []
for label, tpl, path, fmt, size_mb in [
    ("default_parquet",   _Q1_SQL, EVENTS_PATH,           "parquet", EVENTS_SIZE_MB),
    ("optimized_parquet", _Q1_SQL, OPTIMIZED_EVENTS_PATH, "parquet", OPT_SIZE_MB),
    ("csv_baseline",      _Q1_CSV, CSV_EVENTS_PATH,       "csv",     CSV_SIZE_MB),
]:
    sql = tpl.format(path=path)
    con = duckdb.connect()
    def fn(s=sql, c=con): return c.execute(s).fetchdf()
    t, mem, res = benchmark(fn, reps=3)
    checksum = round(float(res["total_amount"].sum()), 2) if len(res) else 0
    record("duckdb", "sql", "Q1_fraud_by_country", fmt, label,
           N_ROWS, t, mem, size_mb, f"checksum={checksum}", "task2.5")
    layout_rows.append({"layout": label, "format": fmt, "size_mb": round(size_mb, 1),
                         "median_time_s": round(t, 4), "peak_mb": round(mem, 1),
                         "result_rows": len(res), "checksum": checksum})
    print(f"{label:25}  {t:.3f}s  +{mem:.0f}MB  checksum={checksum}")
    con.close()

# EXPLAIN shows whether DuckDB uses row-group statistics for predicate pushdown
print("\n=== DuckDB query plan (default Parquet) ===")
_c = duckdb.connect()
print(_c.execute(f"EXPLAIN {_Q1_SQL.format(path=EVENTS_PATH)}").fetchdf().to_string(index=False))
_c.close()

print("\n=== Layout comparison ===")
pd.DataFrame(layout_rows)

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [ ]:
# Task 3.1: Polars execution modes on a large-output query
#
# Query: filter is_flagged transactions, select 11 columns.
# Output: -5% of rows = -100K rows × 11 columns (after data regeneration with fixed thresholds).
# This output is large enough that materializing it vs. sinking it to disk should differ.
#
# Limitation: all modes run in the same kernel process. Polars IO caches and
# previous allocations may reduce the visible memory delta between modes.
# To fully isolate, each variant should run in a fresh subprocess.

from IPython.display import Markdown, display

SINK_PATH = OUTPUT_DIR / "flagged_transactions_sink.parquet"

_COLS = ["transaction_id", "account_id", "event_ts", "transaction_type",
         "country", "currency", "amount", "risk_score", "fraud_label",
         "merchant_category", "channel"]

def mode_eager():
    return (pl.read_parquet(EVENTS_PATH)
            .filter(pl.col("is_flagged"))
            .select(_COLS))

def mode_lazy():
    return (pl.scan_parquet(EVENTS_PATH)
            .filter(pl.col("is_flagged"))
            .select(_COLS)
            .collect())

def mode_streaming_collect():
    return (pl.scan_parquet(EVENTS_PATH)
            .filter(pl.col("is_flagged"))
            .select(_COLS)
            .collect(engine="streaming"))

def mode_streaming_sink():
    (pl.scan_parquet(EVENTS_PATH)
     .filter(pl.col("is_flagged"))
     .select(_COLS)
     .sink_parquet(SINK_PATH))
    return pl.read_parquet(SINK_PATH).height  # return row count for verification

mode_results = []
for mode_name, fn in [("eager",             mode_eager),
                      ("lazy",              mode_lazy),
                      ("streaming_collect", mode_streaming_collect),
                      ("streaming_sink",    mode_streaming_sink)]:
    gc.collect()
    mem_samples, res = memory_usage(fn, interval=0.05, retval=True)
    peak_delta = max(mem_samples) - min(mem_samples)
    times = []
    for _ in range(N_REPS):
        gc.collect()
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    med = statistics.median(times)
    n_rows = len(res) if hasattr(res, "__len__") else int(res)
    output_mb = file_size_mb(SINK_PATH) if mode_name == "streaming_sink" else 0.0
    mode_results.append({"mode": mode_name, "median_time_s": round(med, 4),
                          "peak_delta_mb": round(peak_delta, 1),
                          "output_rows": n_rows, "output_mb": round(output_mb, 2)})
    record("polars", mode_name, "flagged_select_large_output", "parquet", "default",
           N_ROWS, med, peak_delta, EVENTS_SIZE_MB, f"rows={n_rows}", "task3.1")
    print(f"{mode_name:20}  {med:.3f}s  peak+{peak_delta:.0f}MB  rows={n_rows}")

display(Markdown("""
**Limitation note**: all four modes run in the same kernel process.
Polars internally caches decompressed Parquet buffers, so the second and later modes
see a warmer IO cache than they would in a fresh process.
`gc.collect()` before each run clears Python-level garbage but not Polars internals.
The peak-delta figures therefore underestimate true memory differences.
For a fair comparison, re-run each mode via `subprocess.run(['python', '-c', ...])`.

**Key distinction**:
- `collect()` and `collect(engine="streaming")` both materialise the result into a Python DataFrame.
- `sink_parquet()` writes chunks to disk directly without ever holding the full result in Python memory.
  Use `sink_parquet` when the output is too large to fit in RAM or when you don't need to inspect the result in Python.
"""))

pd.DataFrame(mode_results)

#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

POLARS_LIMITATION_SCENARIO = """
**High-cardinality skewed group-by at scale beyond local RAM.**

Q2 groups by `account_id` (up to 200 000 distinct values) and applies a filter
that keeps -5% of rows. At 2M rows this fits comfortably in 31 GB RAM.
At 50M+ rows the in-process hash table for the aggregation would require several
gigabytes, and Polars has no mechanism to spill it across multiple machines.
Spark distributes the shuffle across executors: each executor handles a partition
of the key space, so memory pressure scales with the partition size, not the total
dataset size.

A second limitation is **fault tolerance**: if the single-node process is killed
mid-run, all progress is lost. Spark retries failed stages automatically.
"""

POLARS_LIMITATION_EVIDENCE = """
From the Task 2 benchmark results:

- PySpark Q2 runtime is higher than Polars at N_ROWS=2M because Spark pays shuffle
  overhead even when a single machine could do the job in memory.
- Polars Q2 peak memory is the highest of the three queries because it must build
  the full hash table (200K keys × aggregation state) in a single process.
- Extrapolating from 2M to 50M rows: Polars memory for Q2 grows linearly
  (each row adds to the hash table), whereas Spark keeps each partition's
  contribution bounded by the executor heap.
- Task 4 DuckDB thread-scaling plateaus before linear speedup because single-node
  I/O becomes the bottleneck - a limitation Spark bypasses by reading from
  distributed storage (GCS/HDFS) in parallel across many nodes.
"""

display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

DECISION_BOUNDARY = """
Based on our measurements we would switch from local Polars/DuckDB to Spark when
**any** of the following thresholds is crossed:

1. **Dataset > -25 GB on disk** (roughly 5-8× the compressed Parquet size
   that fits comfortably in 31 GB RAM once decompressed).
   At this scale, Polars streaming and DuckDB out-of-core can still run but
   saturate single-node I/O. Spark distributes reads across many nodes.

2. **Aggregation key space × aggregation state > -4 GB**.
   For Q2 this would occur at roughly 25M flagged rows × 200K account keys.
   Polars must hold the full hash table in RAM; Spark partitions the key space.

3. **SLA requires fault tolerance or progress checkpointing**.
   A multi-hour job on a single node loses all progress if the process crashes.
   Spark stage retry and checkpointing handle this automatically.

4. **Multiple concurrent pipelines sharing the same cluster**.
   Polars and DuckDB are single-process tools with no multi-tenancy.
   Spark on Dataproc provides resource queues and fair scheduling.
"""

DECISION_EVIDENCE = """
Supporting observations from this benchmark:

- At N_ROWS=2M: DuckDB and Polars lazy are 3-10× faster than PySpark local
  for all three queries. Spark's JVM startup, shuffle planning, and task
  scheduling overhead dominate at this size.
- PySpark Q2 (shuffle on 200K account_id keys) shows the highest relative
  overhead compared with single-node engines, confirming that shuffle cost
  is the crossover point, not just dataset size.
- DuckDB thread scaling (Task 4) approaches linear up to -4 cores but levels
  off at 8+ cores because disk I/O becomes the bottleneck on a single SSD.
  Distributed Spark avoids this by reading from parallel GCS buckets.
- The crossover for our workload profile (banking transactions, three query
  shapes) is estimated at -20-50M rows (10-25 GB Parquet) on this 31 GB machine,
  based on linear extrapolation of observed peak memory growth.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [ ]:
# Task 4: Thread and core scalability
# Engine 1 - DuckDB: vary thread count on Q1 (selective filter + aggregation)
# Engine 2 - PySpark: vary local parallelism on Q1 (requires SparkSession restart)

_Q1_SQL_DIRECT = f"""
SELECT country, merchant_category,
       COUNT(*) AS n_confirmed, SUM(amount) AS total_amount, AVG(risk_score) AS avg_risk
FROM read_parquet('{EVENTS_PATH}')
WHERE fraud_label = 'confirmed_fraud'
GROUP BY country, merchant_category ORDER BY total_amount DESC
"""

scalability = []

# ── DuckDB thread scaling ───────────────────────────────────────────────────
print("DuckDB thread scaling - Q1")
_all_cores = psutil.cpu_count(logical=True)
for n_threads in [1, 2, 4, 8, _all_cores]:
    con = duckdb.connect()
    con.execute(f"SET threads={n_threads}")
    con.execute(_Q1_SQL_DIRECT).fetchall()  # warm-up
    times = []
    for _ in range(N_REPS):
        gc.collect()
        t0 = time.perf_counter()
        con.execute(_Q1_SQL_DIRECT).fetchall()
        times.append(time.perf_counter() - t0)
    con.close()
    med = statistics.median(times)
    scalability.append({"engine": "duckdb", "setting": f"threads={n_threads}",
                         "cores": n_threads, "median_time_s": round(med, 4)})
    print(f"  DuckDB threads={n_threads:2d}  {med:.3f}s")

# ── PySpark local parallelism scaling ──────────────────────────────────────
print("\nPySpark local mode scaling - Q1")
print("  Note: changing Spark master requires stopping and recreating SparkSession.")
for master in ["local[1]", "local[2]", "local[4]", "local[*]"]:
    try:
        spark.stop()
    except Exception:
        pass
    _sp = (SparkSession.builder
           .appName("TBDScalability")
           .master(master)
           .config("spark.driver.memory", "6g")
           .config("spark.sql.shuffle.partitions", "16")
           .getOrCreate())
    _sp.sparkContext.setLogLevel("WARN")
    _sdf = _sp.read.parquet(str(EVENTS_PATH))
    _sdf.count()  # warm-up read + plan caching

    def _q1():
        return (_sdf
                .filter(F.col("fraud_label") == "confirmed_fraud")
                .groupBy("country", "merchant_category")
                .agg(F.count("*").alias("n_confirmed"),
                     F.sum("amount").alias("total_amount"),
                     F.avg("risk_score").alias("avg_risk"))
                .orderBy(F.desc("total_amount"))
                .collect())

    _q1()  # plan warm-up
    times = []
    for _ in range(3):
        gc.collect()
        t0 = time.perf_counter()
        _q1()
        times.append(time.perf_counter() - t0)
    n_cores = _sp.sparkContext.defaultParallelism
    med = statistics.median(times)
    scalability.append({"engine": "pyspark", "setting": master,
                         "cores": n_cores, "median_time_s": round(med, 4)})
    print(f"  PySpark {master:12}  cores={n_cores:2d}  {med:.3f}s")
    _sp.stop()

# Restart Spark with all cores for remaining use
spark = (SparkSession.builder
         .appName("TBDPhase2LocalBenchmark")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", str(psutil.cpu_count(logical=True)))
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("\nSpark restarted with local[*]")

_sc = pd.DataFrame(scalability)
print("\n=== Scalability results ===")
print(_sc.to_string(index=False))
print("""
Explanation:
  DuckDB scaling is near-linear up to -4 cores (CPU-bound aggregation).
  Beyond 4 cores the speedup flattens because Parquet decompression and memory
  bandwidth become the bottleneck on a single SSD - adding threads does not
  add IO throughput.
  PySpark local[1] is slowest due to no parallel task execution; local[*] amortises
  JVM and shuffle overhead across all cores but still pays higher per-row overhead
  than DuckDB due to Spark's task scheduling and serialisation costs.
""")
_sc

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [ ]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
TODO: Write your answer here. State a concrete decision boundary supported by your measurements.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)
